# Every way to specify a systematic with `graphed.vary`

A guided tour, simplest to most complex. Every cell is executed and prints the
`graphed.labels()` / `graphed.points()` / `graphed.variations()` / `graphed.explain()` it
actually produces — no number
or label below is asserted from memory.

The notebook is self-contained: toy numpy/awkward arrays and a tiny inline correctionlib set built
in the next cell, and the one level that reads a ROOT file under a process pool writes that file
first. Every output below is committed from an actual run; to re-run them yourself,
`pip install "graphed[awkward,numpy]" graphed-histogram graphed-executors correctionlib uproot`
and execute the cells in order.

> **Companion — the full grid on real data.** The [ADL benchmark](https://github.com/graphed-org/coffea-benchmarks-graphed-mvp/blob/graphed-mvp/graphed-adl-benchmarks.ipynb) runs the same JES + b-tag setup on the real 50k skim: because the b-tag SF is computed on the JES-varied jets, one plain `graphed.vary` per nuisance auto-fans to the **full 15-universe grid** — propagation for free (level 9). The datacard's seven-template union is one `composes_as_union=True` away. This tour is the deep dive into that mechanism: auto-fanout, its prune, its guard, and the off-grid placements (Levels 15-18) that name universes by hand.

## The map

An analyst makes three orthogonal choices, and this tour walks their product plus the three ways
universes can relate.

**(a) What is varied.** A bare `Array` (the *loose* form), an event context's **weight**
(`is_weight=True` + `points=`), or an event context's **collections** (`collections=`). All
three mint the same label grammar `f"{name}_{tag}"` and the same default point `{name: tag}`.

**(b) How many nuisance families.** One call; a second call extending the *same* family; or
independent families, which compose as the **union**, never the cross product.

**(c) How universes relate.** Three genuinely distinct mechanisms, which the word "correlated"
conflates and which this notebook teaches apart:

| Analyst intent | Mechanism | Level |
|---|---|---|
| Two registrations are the same fit parameter | **name identity** — share the nuisance `name` | 8 |
| A correction is a function of a quantity a nuisance moves | **propagation** — compute it from the varied quantity | 9 |
| A variation computed over another nuisance's varied nodes | **auto-fanout** — the joint grid is minted automatically | 10+ |
| A universe named by hand at a prescribed coordinate | **off-grid placement** — re-point a member onto a chosen point | 15+ |

> **The second verb — placement (Levels 15–18).** Everything through Level 14 *declares* members and lets `graphed` derive their universes. Levels 15–18 add *placement*: name a coordinate for a member by hand — the μR×μF 7-point set, prescribed PDF-eigenvector-style directions. Declares and placements share one `points=` list; the structure of each entry — a `(tag, array)` tuple versus a `{nuisance: coordinate}` map — picks the verb, and misuse raises `graphed.PointError` (Level 18).

> **The instrument (Level 20).** `graphed.explain(ctx)` reports how the calls you wrote became the universes: how each family entered the ambient weight, what the ambient is made of here, and where every universe came from. The level uses it to find two mistakes that print a perfectly ordinary label set.

> **The capstone (Level 19).** The CMS JEC → JES/JER → Type-1 MET → b-tag stack, mocked end to end on one context: every coupling above appears at once, each universe is read back against the prescription, and the section closes with the one place a `points=` entry was needed.

## Vocabulary

- **universe** — one systematic variation: the analysis evaluated at one point in nuisance space. The many-universes picture is standard HEP diction — the neutrino *multisim* method ([MiniBooNE, Phys. Rev. D 79, 072002 (2009)](https://arxiv.org/abs/0806.1449)), echoed in the ATLAS/CMS *bootstrap*, where each replica is a dataset from a "parallel universe" ([ATL-PHYS-PUB-2021-011](https://cds.cern.ch/record/2759945)).
- **nuisance** — a family name; the nuisance parameter it becomes in the fit.
- **coordinate** — the per-axis displacement (HistFactory's α, combine's θ).
- **point** — a sparse `{nuisance: coordinate}` map. HS3 calls this a *parameter point*.
- **correlated** — shares a nuisance name. Nothing else.
- **propagation** — a correction depends on a quantity a nuisance moves. *Not* correlation.
- **one-at-a-time set** — the axis-aligned unit points; what a datacard normally wants.
- **factorization error** — the non-factorizable part (factorization non-closure) a joint universe measures.

**Every universe is a point in nuisance space; a label is a NAME for that point; resolution
projects the requested point onto the axes a container knows and then falls back to nominal.**
`nominal` is the origin: every coordinate at 0.

## Level 0 — setup

Two toy datasets and one toy correctionlib set. Nothing else is imported.

- `ctx0()` — a numpy-backed event context with jet collections `pt`, `eta`, `mass` and a muon collection `mu_pt`. Used wherever the
  level is about *declaration shape* rather than physics.
- `toy_jets()` — jagged jet pT, an awkward array, used where a correction has to be evaluated.
- `TOY_SF` — a correctionlib v2 set whose `up`/`down` uncertainty **grows with pT**
  (2% / 5% / 10% / 20% in the pT bins `[0,30,60,100,∞)`). That pT dependence is one of the two
  sources of the factorization error measured in level 12; a scale factor flat in pT with the selection frozen at nominal makes that error vanish — level 12 runs exactly that leg as its
  control.

In [1]:
import json

import awkward as ak
import correctionlib
import numpy as np

import graphed
from graphed import Session
from graphed.awkward import AwkwardBackend, from_awkward, gak
from graphed.context import EventContext
from graphed.numpy import NumpyBackend, from_record


def ctx0():
    """A fresh Session + a 12-event context: jet `pt`/`eta`/`mass` and a muon `mu_pt`."""
    s = Session(NumpyBackend())
    r = from_record(
        s,
        "ev",
        pt=np.arange(1.0, 13.0),
        eta=np.arange(1.0, 13.0) / 10.0,
        mass=np.arange(1.0, 13.0) / 8.0,
        mu_pt=np.arange(20.0, 32.0),
    )
    return s, EventContext(s, r["pt"], collections={k: r[k] for k in ("pt", "eta", "mass", "mu_pt")})


def toy_jets(n_events=2000, seed=7):
    """Jagged jet pT: 2-5 jets per event, exponential pT, straddling the SF bin edges."""
    rng = np.random.default_rng(seed)
    counts = rng.integers(2, 6, size=n_events)
    pt = rng.exponential(45.0, size=int(counts.sum())) + 10.0
    return ak.unflatten(pt, counts)


def toy_session():
    s = Session(AwkwardBackend())
    return s, from_awkward(s, "jet_pt", toy_jets())


TOY_SF = json.dumps(
    {
        "schema_version": 2,
        "description": "a pT-binned b-tag-like scale factor; the uncertainty grows with pT",
        "corrections": [
            {
                "name": "toy_sf",
                "version": 1,
                "inputs": [{"name": "systematic", "type": "string"}, {"name": "pt", "type": "real"}],
                "output": {"name": "sf", "type": "real"},
                "data": {
                    "nodetype": "category",
                    "input": "systematic",
                    "content": [
                        {
                            "key": "central",
                            "value": {
                                "nodetype": "binning",
                                "input": "pt",
                                "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                                "content": [1.00, 1.00, 1.00, 1.00],
                                "flow": "clamp",
                            },
                        },
                        {
                            "key": "up",
                            "value": {
                                "nodetype": "binning",
                                "input": "pt",
                                "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                                "content": [1.02, 1.05, 1.10, 1.20],
                                "flow": "clamp",
                            },
                        },
                        {
                            "key": "down",
                            "value": {
                                "nodetype": "binning",
                                "input": "pt",
                                "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                                "content": [0.98, 0.95, 0.90, 0.80],
                                "flow": "clamp",
                            },
                        },
                    ],
                },
            }
        ],
    }
).encode()

SF = correctionlib.CorrectionSet.from_string(TOY_SF.decode())["toy_sf"]


def show(container, *, variations=False):
    """labels(), then points() one per line, then optionally variations()."""
    print("labels    :", graphed.labels(container))
    pts = graphed.points(container)
    print("points    :")
    for label in graphed.labels(container):
        print(f"    {label:22s} {pts[label]}")
    if variations:
        print("variations:", graphed.variations(container))


print(
    "correctionlib",
    correctionlib.__version__,
    "| toy SF at pT=25/50/80/200, systematic=up:",
    [round(SF.evaluate("up", p), 2) for p in (25.0, 50.0, 80.0, 200.0)],
)

correctionlib 2.9.0 | toy SF at pT=25/50/80/200, systematic=up: [1.02, 1.05, 1.1, 1.2]


## Level 1 — one at a time, on a bare array

The simplest systematic: two keyword tags on an `Array`. The label is `f"{name}_{tag}"` and its
point is the axis-aligned `{name: tag}` — every other nuisance sits at 0, which is what absence
means. `nominal` maps to the empty point, the origin.

This is the *loose* form: no event context, no weight, just a varied quantity.

In [2]:
s, c = ctx0()
pt = c["pt"]

jes = graphed.vary(pt, "jes", up=pt * 1.1, down=pt * 0.9)
show(jes)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}


## Level 2 — extending a family

A nuisance family is open. A second `graphed.vary` call with the *same* name adds tags to it, so
the fit still sees **one** parameter. Three tags on one axis, not three axes.

In [3]:
jes2 = graphed.vary(jes, "jes", up2=graphed.nominal(jes) * 1.21)
show(jes2)

labels    : ('nominal', 'jes_up', 'jes_down', 'jes_up2')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    jes_up2                {'jes': 'up2'}


## Level 3 — a weight systematic

`is_weight=True` on an event context: the selection and the observable are identical in every
universe, only the per-event weight moves. `graphed.variations()` reports the kind string
`Kind.WEIGHT`.

In [4]:
s, c = ctx0()
w = c["pt"] * 0.5

btag = graphed.vary(c, "btag", w, is_weight=True, up=w * 1.2, down=w * 0.8)
show(btag, variations=True)

labels    : ('nominal', 'btag_up', 'btag_down')
points    :
    nominal                {}
    btag_up                {'btag': 'up'}
    btag_down              {'btag': 'down'}
variations: {'btag': {'up': (Kind.WEIGHT, None), 'down': (Kind.WEIGHT, None)}}


## Level 4 — a shift systematic

`collections=` instead: the *kinematics* move, so the selection and the observable both change.
Same declaration shape, same label grammar, same default points — but `graphed.variations()`
reports `Kind.SHIFT`, and downstream every cut is re-evaluated per universe.

In [5]:
s, c = ctx0()
pt = c["pt"]

shift = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
show(shift, variations=True)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
variations: {'jes': {'up': (Kind.SHIFT, None), 'down': (Kind.SHIFT, None)}}


## Level 5 — lockstep: one nuisance, two collections

One nuisance can move several collections **coherently** — a jet-energy scale rescales the jet
four-vector, so `pt` and `mass` move by the **same** factor while `eta` is untouched. The universe
count does **not** grow: still three labels, because it is still one axis.


In [6]:
s, c = ctx0()
pt, mass = c["pt"], c["mass"]

up, down = 1.1, 0.9  # one factor per direction, applied to every collection the axis moves
lock = graphed.vary(
    c,
    "jes",
    collections={
        "pt": {"up": pt * up, "down": pt * down},
        "mass": {"up": mass * up, "down": mass * down},
    },
)
show(lock)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}


## Level 6 — stacked independent families compose as the UNION

Two independent 2-tag families give **5** universes (`1 + 2 + 2`), not 9. A label registered
with a bare declare (no placement) differs from nominal on exactly **one** axis, so no cross product
can arise implicitly. This axis-aligned set is the one-at-a-time set the datacard wants.

**The nominal names the factor being varied.** A weight registration's third argument is not just a
starting value: it says *which* factor of the ambient weight this family moves. `graphed` compares it
**by node** — never by value, so a re-computed expression with equal values is a different thing —
and decides one of three outcomes. `mu`'s nominal below is the `graphed.weight()` handle, the whole
composition, so `mu` is a **relative delta**: its members *are* the ambient rescaled and they
**replace** the running product at their own labels instead of multiplying into it, which is why
`mu_up` is 1.05 × the nominal weight. A nominal that is a weight already registered as a factor —
the b-tag SF a second flavour source also varies — **joins** that factor: one SF in the product,
its members the universes of that one operation. Anything else is a new factor. Level 20 prints
which outcome each call got.

> The labels and points below are what they always were; the values changed when this rule landed.
> Before it, the handle was appended as one more factor, so this cell's nominal weight was the b-tag
> weight **squared** and every `mu` universe was 1.05 × that square.


In [7]:
ambient = graphed.weight(btag)  # the btag family from level 3
stacked = graphed.vary(btag, "mu", ambient, is_weight=True, up=ambient * 1.05, down=ambient * 0.95)
show(stacked, variations=True)
print()
print("2 tags + 2 tags ->", len(graphed.labels(stacked)), "universes, not", 3 * 3)

labels    : ('nominal', 'btag_up', 'btag_down', 'mu_up', 'mu_down')
points    :
    nominal                {}
    btag_up                {'btag': 'up'}
    btag_down              {'btag': 'down'}
    mu_up                  {'mu': 'up'}
    mu_down                {'mu': 'down'}
variations: {'btag': {'up': (Kind.WEIGHT, None), 'down': (Kind.WEIGHT, None)}, 'mu': {'up': (Kind.WEIGHT, None), 'down': (Kind.WEIGHT, None)}}

2 tags + 2 tags -> 5 universes, not 9


## Level 7 — shift then weight

A kinematic family and a weight family stack the same way. Note the second line: the **ambient
weight** carries the inherited `jes` labels even though `jes` is not one of *its* registered
families — a container's labels legitimately outrun its own tag map. That is why axis sets are read from the Session registry, not derived from the tags.

In [8]:
w7 = shift["pt"] * 0.5  # the shift context from level 4
both = graphed.vary(shift, "btag", w7, is_weight=True, up=w7 * 1.2, down=w7 * 0.8)

print("context labels:", graphed.labels(both))
print("weight  labels:", graphed.labels(graphed.weight(both)))
print("variations    :", graphed.variations(both))

context labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'btag_up__jes_up', 'btag_up__jes_down', 'btag_down__jes_up', 'btag_down__jes_down')
weight  labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'btag_up__jes_up', 'btag_up__jes_down', 'btag_down__jes_up', 'btag_down__jes_down')
variations    : {'btag': {'up': (Kind.WEIGHT, None), 'down': (Kind.WEIGHT, None)}, 'jes': {'up': (Kind.SHIFT, None), 'down': (Kind.SHIFT, None)}}


## Level 8 — mechanism 1: NAME IDENTITY

The first correlation mechanism. Registering **one nuisance name** as both a shift and a weight
ties them into a single fit parameter — combine's rule: *"multiple instances of any
nuisance parameter, sharing the same name, are treated as a single parameter."*

The label set does **not** grow: one universe carries both effects. `graphed.variations()` reports
the union `Kind.WEIGHT|SHIFT` for the dual tag, while the shift-only tag stays `Kind.SHIFT`.

In [9]:
s, c = ctx0()
pt = c["pt"]

sh = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
w8 = sh["pt"] * 0.5
dual = graphed.vary(sh, "jes", w8, is_weight=True, up=w8 * 1.3)

print("labels    :", graphed.labels(dual), "  <- still three; jes_up now moves BOTH")
print("variations:", graphed.variations(dual))

labels    : ('nominal', 'jes_up', 'jes_down')   <- still three; jes_up now moves BOTH
variations: {'jes': {'up': (Kind.WEIGHT|SHIFT, None), 'down': (Kind.SHIFT, None)}}


## Level 9 — mechanism 2: PROPAGATION

The second mechanism, and the one most often mislabelled "correlation". The toy scale factor is a
**function of jet pT**, and the `jes` nuisance moves pT. Inside the `jes_up` universe the scale
factor must be re-evaluated on the *shifted* jets.

`gak.apply_correction` maps over a `Varied`: hand it the **varied** pT and it returns a `Varied`
over the same labels, with a **distinct node per universe** — the correction genuinely re-evaluated
on each universe's own jets. Nothing declares this; it falls out of passing the varied quantity.

(The older workaround of fanning out by hand over universes is obsolete: `apply_correction`'s plan
is now pickled and runs across a process pool, which level 13 proves.)

In [10]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)

sf_central = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate, args=["central", "$0"])

print("type:", type(sf_central).__name__, " labels:", graphed.labels(sf_central))
for label in graphed.labels(sf_central):
    print(f"    {label:10s} -> IR node {graphed.universe(sf_central, label).node_id}")
print()
print("propagation is not correlation: `jes` is the only registered nuisance;")
print("the SF is simply a function of a quantity `jes` moves.")

type: Varied  labels: ('nominal', 'jes_up', 'jes_down')
    nominal    -> IR node 3
    jes_up     -> IR node 4
    jes_down   -> IR node 5

propagation is not correlation: `jes` is the only registered nuisance;
the SF is simply a function of a quantity `jes` moves.


## Level 10 — the joint grid, minted automatically

The third mechanism, and the important one. When a variation is *computed over* another nuisance's
varied nodes, it genuinely **depends on** that axis, and `graphed` mints the full joint grid on its
own — no placement, no manual enumeration.

Below, `sf` is the b-tag-like weight built off the **JES-varied** pT (the propagation of level 9). So
a single plain `graphed.vary(central, "sf", up=…, down=…)` does not give five universes — it gives
**nine**: `nominal`, the two `jes`, the two one-at-a-time `sf`, and the **four joint cross-terms**
`sf_up__jes_up … sf_down__jes_down`.

- the joint label is **machine-minted** as `f"{name}_{tag}__{foreign}_{ftag}"`; you never spell it;
- its **point** names both axes (`{sf: up, jes: up}`) and binds the **real cross node** the graph
  already holds — the b-tag SF evaluated on *that* universe's shifted pT;
- this is the defect this work removes: before auto-fanout the foreign `jes` coordinate collapsed
  to nominal, silently discarding the cross-term.

The full grid imposes no analysis choice — it is every universe the dependency implies. Level 14 shows the three knobs that mold it.

In [11]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0  # a selection on the SHIFTED pT
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    """Per-jet SF off the VARIED pT (level 9), multiplied over the selected jets."""
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate, args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


# ONE plain vary -- no placements. central/up/down are each computed over the jes-varied pT, so the sf
# family DEPENDS on the jes axis, and graphed mints the full jes x sf grid automatically.
weight = graphed.vary(sf("central"), "sf", up=sf("up"), down=sf("down"))

show(weight)
print()
print(len(graphed.labels(weight)), "universes = 1 nominal + 2 jes + 2 sf + 4 joints.")
print("each joint label is machine-minted 'sf_<tag>__jes_<tag>'; its point names BOTH axes and")
print("binds the real cross node -- the SF on THAT universe's shifted pT, never the nominal one.")

labels    : ('nominal', 'jes_up', 'jes_down', 'sf_up', 'sf_down', 'sf_up__jes_up', 'sf_up__jes_down', 'sf_down__jes_up', 'sf_down__jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    sf_up                  {'sf': 'up'}
    sf_down                {'sf': 'down'}
    sf_up__jes_up          {'jes': 'up', 'sf': 'up'}
    sf_up__jes_down        {'jes': 'down', 'sf': 'up'}
    sf_down__jes_up        {'jes': 'up', 'sf': 'down'}
    sf_down__jes_down      {'jes': 'down', 'sf': 'down'}

9 universes = 1 nominal + 2 jes + 2 sf + 4 joints.
each joint label is machine-minted 'sf_<tag>__jes_<tag>'; its point names BOTH axes and
binds the real cross node -- the SF on THAT universe's shifted pT, never the nominal one.


## Level 11 — why a joint label executes: resolution by PROJECTION

The auto-minted joints cost almost nothing because they reuse nodes that already exist. A joint label
is asked of containers that know only *some* of its axes; each contributes its member at the
**projection** of the point onto the axes it carries, then falls back to nominal.

Below, the b-tag weight is built over the JES-varied jets, so `graphed.weight(b)` auto-fans to the
`btag × jes` grid. The observable `obs` carries the `jes` axis only:

- `jes_up` → its own shifted member;
- `btag_up` → an axis `obs` does not carry, so **nominal**;
- `btag_up__jes_up` → the point `{btag: up, jes: up}` restricted to `{jes}` is `{jes: up}`, so `obs`
  gets the **shifted** member.

The weight, which carries both axes, returns the registered joint value. This is the whole of the
resolution machinery, and it is why a joint universe costs only the interning of nodes that already
exist.

In [12]:
s, c = ctx0()
pt = c["pt"]
a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
wv = a["pt"] * 0.5  # an ambient weight off the jes-varied pT
b = graphed.vary(a, "btag", wv, is_weight=True, up=wv * 1.2, down=wv * 0.8)
# b's btag members are computed over the jes-varied wv, so weight(b) auto-fans to the btag x jes grid.
weight = graphed.weight(b)
print("weight labels:", graphed.labels(weight))

obs = a["pt"]  # knows the `jes` axis ONLY
print()
for label in ("nominal", "jes_up", "btag_up", "btag_up__jes_up"):
    m = graphed.member_of(obs, label)
    print(f"  obs[{label:16s}] node {m.node_id}  {[float(x) for x in list(s.materialize(m))[:3]]}")

print()
print(
    "weight[btag_up__jes_up]:",
    [float(x) for x in list(s.materialize(graphed.universe(weight, "btag_up__jes_up")))[:3]],
)

weight labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'btag_up__jes_up', 'btag_up__jes_down', 'btag_down__jes_up', 'btag_down__jes_down')

  obs[nominal         ] node 1  [1.0, 2.0, 3.0]
  obs[jes_up          ] node 5  [1.1, 2.2, 3.3000000000000003]
  obs[btag_up         ] node 1  [1.0, 2.0, 3.0]
  obs[btag_up__jes_up ] node 5  [1.1, 2.2, 3.3000000000000003]

weight[btag_up__jes_up]: [0.66, 1.32, 1.98]


## Level 12 — what a joint universe MEASURES: the factorization error

A joint universe is **not** "covering the correlation" — the fit already correlates by name
(level 8), and HistFactory's response is a *sum of one-dimensional terms*
(`A = nominal + Σ_i I_i(α_i)`) with no slot for a joint template. What a joint universe measures is
the **error** of that factorization: how far the true two-axis response sits from the factorized 1-D prediction `jes_1D + sf_1D − nominal`.

Three legs, and it is their **ordering** that carries the lesson, not any absolute number:

| leg | SF pT-dependent? | selection on shifted pT? | expected |
|---|---|---|---|
| the teaching case | yes | yes | largest |
| migration only | no | yes | smaller, still nonzero |
| **positive control** | no | no | **0**, to machine precision |

The third leg is the control that proves the instrument reads zero when there is nothing to
measure. Built that way — a pT-*flat* scale factor with a frozen selection — a joint universe would read zero and teach the opposite of the intended lesson.

In [13]:
def yields(pt_dependent, live_selection):
    s = Session(AwkwardBackend())
    raw = from_awkward(s, "jet_pt", toy_jets())
    jes_pt = graphed.vary(raw, "jes", up=raw * 1.05, down=raw * 0.95)
    keep = (jes_pt if live_selection else raw) > 30.0
    passes = gak.sum(keep, axis=1) >= 2

    def sf(systematic):
        arg = jes_pt if pt_dependent else (jes_pt * 0.0 + 50.0)  # freeze the SF's pT argument
        per_jet = gak.apply_correction(TOY_SF, "toy_sf", [arg], SF.evaluate, args=[systematic, "$0"])
        return gak.prod(per_jet[keep], axis=1) * passes

    # plain vary: the four joints sf_<tag>__jes_<tag> are minted by auto-fanout -- no placements.
    w = graphed.vary(sf("central"), "sf", up=sf("up"), down=sf("down"))
    return {L: float(ak.sum(s.materialize(graphed.universe(w, L)))) for L in graphed.labels(w)}


def factorization_error(title, tot):
    print(title)
    base = tot["nominal"]
    worst = 0.0
    for jes_label in ("jes_up", "jes_down"):
        for tag in ("up", "down"):
            joint = tot[f"sf_{tag}__{jes_label}"]  # the auto-minted cross-term
            linear = tot[jes_label] + tot[f"sf_{tag}"] - base  # the fit's 1-D sum
            err = joint - linear
            worst = max(worst, abs(err) / base)
            print(
                f"    {jes_label:8s} x sf_{tag:5s}: joint={joint:12.6f}  1D-sum={linear:12.6f}"
                f"  error={err:+11.6f}  ({100 * err / base:+.4f}% of nominal)"
            )
    print(f"    -> largest |error| = {100 * worst:.4f}% of nominal\n")
    return worst


w1 = factorization_error("pT-binned SF, live selection  (the teaching case)", yields(True, True))
w2 = factorization_error(
    "CONTROL  pT-flat SF, live selection  (selection migration only)", yields(False, True)
)
w3 = factorization_error(
    "CONTROL  pT-flat SF, selection frozen at nominal  (must read 0)", yields(False, False)
)

print(
    f"ordering holds: {w1 > w2 > w3}"
    f"   (pT-binned+live {100 * w1:.4f}%  >  migration-only {100 * w2:.4f}%  >  control {w3:.2e})"
)
print("control is zero to machine precision:", w3 < 1e-9)

pT-binned SF, live selection  (the teaching case)
    jes_up   x sf_up   : joint= 1974.875095  1D-sum= 1942.498495  error= +32.376601  (+2.2130% of nominal)
    jes_up   x sf_down : joint= 1127.367392  1D-sum= 1150.617130  error= -23.249738  (-1.5892% of nominal)
    jes_down x sf_up   : joint= 1828.447268  1D-sum= 1859.498495  error= -31.051226  (-2.1224% of nominal)
    jes_down x sf_down : joint= 1089.826382  1D-sum= 1067.617130  error= +22.209251  (+1.5181% of nominal)
    -> largest |error| = 2.2130% of nominal

CONTROL  pT-flat SF, live selection  (selection migration only)
    jes_up   x sf_up   : joint= 1730.679280  1D-sum= 1721.590460  error=  +9.088820  (+0.6212% of nominal)
    jes_up   x sf_down : joint= 1305.432745  1D-sum= 1313.285928  error=  -7.853183  (-0.5368% of nominal)
    jes_down x sf_up   : joint= 1629.570521  1D-sum= 1638.590460  error=  -9.019939  (-0.6165% of nominal)
    jes_down x sf_down : joint= 1238.010241  1D-sum= 1230.285928  error=  +7.724314  (+0.528

CONTROL  pT-flat SF, selection frozen at nominal  (must read 0)
    jes_up   x sf_up   : joint= 1677.590460  1D-sum= 1677.590460  error=  +0.000000  (+0.0000% of nominal)
    jes_up   x sf_down : joint= 1269.285927  1D-sum= 1269.285928  error=  -0.000000  (-0.0000% of nominal)
    jes_down x sf_up   : joint= 1677.590460  1D-sum= 1677.590460  error=  +0.000000  (+0.0000% of nominal)
    jes_down x sf_down : joint= 1269.285927  1D-sum= 1269.285928  error=  -0.000000  (-0.0000% of nominal)
    -> largest |error| = 0.0000% of nominal

ordering holds: True   (pT-binned+live 2.2130%  >  migration-only 0.6212%  >  control 1.55e-16)
control is zero to machine precision: True


## Level 13 — the same program through a ROOT file, under a process pool

Two things at once.

**Propagation crosses a process boundary.** `gak.apply_correction` records an External node, and the
plan below pickles and runs on a persistent 4-worker pool — the auto-fanned grid and all — with no
hand fan-out and no closure to ship. The ROOT file is written at the top of the cell, so the tour
stands alone — read back with `uproot.graphed`, the program is the one you would write on a real
skim.

**The datacard control.** The default is the full nine-universe grid; `composes_as_union=True`
collapses it back to the pre-fanout datacard set (nominal, the two `jes`, the two one-at-a-time
`sf`). The four joints are exactly the cross-terms the old collapse dropped silently — here they are
present by default and measurable, and opting out is one keyword, not a hand-built fan-out you can
forget. Always read `graphed.points()` to see which universes a program actually carries.

In [14]:
import tempfile

import graphed_histogram as gh
import hist.graphed as hg
import uproot
from graphed_executors.local import ProcessPoolExecutor

skim = f"{tempfile.mkdtemp()}/jets.root"
uproot.recreate(skim)["Events"] = {"Jet_pt": toy_jets(n_events=4000, seed=13)}


def real_program(compose):
    g = uproot.graphed(f"{skim}:Events", library="ak")
    raw = g.Jet_pt
    jes_pt = graphed.vary(raw, "jes", up=raw * 1.05, down=raw * 0.95)
    keep = jes_pt > 30.0
    passes = gak.sum(keep, axis=1) >= 2

    def sf(systematic):
        per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate, args=[systematic, "$0"])
        return gak.prod(per_jet[keep], axis=1) * passes

    # default: auto-fanout mints the 9-universe grid. compose: collapse back to the datacard union.
    weight = graphed.vary(sf("central"), "sf", up=sf("up"), down=sf("down"), composes_as_union=compose)
    h = (
        hg.Hist.new.Reg(1, 0.0, 1e9, name="ht")
        .Double()
        .fill(ht=gak.sum(jes_pt[keep], axis=1), weight=[weight])
    )
    return gh.plan({"h": h}, steps_per_file=8, backend="graphed.awkward:AwkwardBackend"), weight


executor = ProcessPoolExecutor(max_workers=4, persistent=True)
try:
    plan, weight = real_program(compose=False)
    res = executor.run(plan)
    tot = {k: float(v.sum()) for k, v in gh.unpack(res.value)["h"].items()}
    print(
        "ProcessPoolExecutor: OK -",
        res.n_partitions,
        "partitions, ",
        len(tot),
        "universes; apply_correction survived the pool",
    )
    print("points['sf_up__jes_up'] =", graphed.points(weight)["sf_up__jes_up"])
    for k, v in tot.items():
        print(f"    {k:16s} {v:14.6f}")
    real = factorization_error("\nfactorization error on the jets read back", tot)

    plan_c, weight_c = real_program(compose=True)  # composes_as_union -> the datacard union
    tot_c = {k: float(v.sum()) for k, v in gh.unpack(executor.run(plan_c).value)["h"].items()}
    print("CONTROL composes_as_union=True:", len(tot_c), "universes =", list(graphed.labels(weight_c)))
    print("   the four joints are gone -- the pre-fanout datacard set. The default kept them, so the")
    print("   factorization error above is measurable rather than dropped on the floor.")
finally:
    executor.close()

ProcessPoolExecutor: OK - 8 partitions,  9 universes; apply_correction survived the pool
points['sf_up__jes_up'] = {'jes': 'up', 'sf': 'up'}
    nominal             2916.000000
    jes_up              3021.000000
    jes_down            2820.000000
    sf_up               3778.514966
    sf_down             2206.693072
    sf_up__jes_up       3950.504380
    sf_up__jes_down     3615.884473
    sf_down__jes_up     2263.416045
    sf_down__jes_down    2158.599589

factorization error on the jets read back
    jes_up   x sf_up   : joint= 3950.504380  1D-sum= 3883.514966  error= +66.989414  (+2.2973% of nominal)
    jes_up   x sf_down : joint= 2263.416045  1D-sum= 2311.693072  error= -48.277027  (-1.6556% of nominal)
    jes_down x sf_up   : joint= 3615.884473  1D-sum= 3682.514966  error= -66.630493  (-2.2850% of nominal)
    jes_down x sf_down : joint= 2158.599589  1D-sum= 2110.693072  error= +47.906518  (+1.6429% of nominal)
    -> largest |error| = 2.2973% of nominal

CONTROL composes_a

## Level 14 — controlling the grid: union, prune, and the loud guard

The default fanout is the **complete grid**, with no analysis choice imposed. Three knobs mold it, coarse to fine:

- **`composes_as_union=True`** — collapse to the one-at-a-time datacard union (the pre-fanout
  behaviour), byte-for-byte. This is an *analysis choice* — "a datacard is one-at-a-time" — applied
  explicitly, not the framework default.
- **a placement** (`points=[…, {nuisance: coordinate}]`) — prune the grid to a chosen subset of
  coordinate maps (here the diagonal). Over the *dependent* `sf` family it is **reachability-validated**:
  every coordinate must name a universe the grid derives. (Level 15 shows the same syntax re-pointing
  an *independent* member off the grid, and carries custom numeric coordinates too.)
- **`max_universes=`** — the loud guard. A default grid whose family sizes multiply past the budget
  (default 64) is refused *before* it is minted, naming the count and the families. `composes_as_union`
  and an explicit placement are never guarded — you asked for exactly those universes.

A dependent *chain* (jes → btag → pu) does not explode to a full cross product: each family fans only
over the shared root axis, so three families give fifteen universes, not twenty-seven. The guard is
there for the genuinely wide grids, and for turning a runaway into a one-line placement selection.

In [15]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate, args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


variations = {"up": sf("up"), "down": sf("down")}

# 1. the default -- the full grid, no analysis choice imposed
full = graphed.vary(sf("central"), "sf", points=variations)
print("default full grid :", len(graphed.labels(full)), "universes")

# 2. composes_as_union=True -- collapse to the one-at-a-time datacard set (the pre-fanout behaviour)
union = graphed.vary(sf("central"), "sf", points=variations, composes_as_union=True)
print("composes_as_union :", len(graphed.labels(union)), "universes", list(graphed.labels(union)))

# 3. placements -- mix {nuisance: coordinate} maps into points= to prune the grid to a chosen
#    subset (here the diagonal). Over the dependent sf family this is reachability-validated.
diag = graphed.vary(
    sf("central"),
    "sf",
    points=[*variations.items(), {"sf": "up", "jes": "up"}, {"sf": "down", "jes": "down"}],
)
print(
    "placement prune   :",
    len(graphed.labels(diag)),
    "universes",
    [L for L in graphed.labels(diag) if "__" in L],
)

# 4. max_universes -- the loud guard, fires BEFORE a runaway grid is minted
try:
    graphed.vary(sf("central"), "sf", points=variations, max_universes=8)
except graphed.GraphedError as e:
    print("max_universes=8   : refused ->", e)

default full grid : 9 universes
composes_as_union : 5 universes ['nominal', 'jes_up', 'jes_down', 'sf_up', 'sf_down']
placement prune   : 7 universes ['sf_up__jes_up', 'sf_down__jes_down']
max_universes=8   : refused -> graphed.vary('sf') would fan out to 9 universes (jes(3) x sf(3)); pass points= placements to select a subset, or raise max_universes (currently 8)


## Level 15 — placing a member off the grid: the additive re-point

Levels 1–14 use one verb: **declare** — give an array a label, whose point is the axis-aligned default
`{name: tag}` (Level 14's prune only *selects* among points the fanout already derives). The second
verb is **place**: hand `graphed.vary` a `{nuisance: coordinate}` map instead of a `(tag, array)`
tuple, and it names a coordinate for an already-declared label.

Both verbs share one `points=` list, and the *structure* of each entry picks the verb:

| entry | verb |
|---|---|
| `("a", array)` | **declare** the member `corr_a` |
| `{"corr": "a", "jes": "up", "mus": "up"}` | **place** the label `corr_a` at that point |

What a placement *means* depends on whether the named member genuinely depends on the foreign axes:

- over a member **built from** a varied axis (Level 14's `sf` off the jes-varied jets) it is a
  **prune** — select a joint the auto-fanout already derives, keeping the member's own axis;
- over an **independent** member (read off `graphed.nominal`, carrying no foreign axis) it is an
  **additive re-point** — the label leaves its one-at-a-time default and lands on the prescribed
  foreign-only point, and its **own axis is dropped** (the label is a name for the point, not a cross).

Below, `corr_a` is independent of `jes`/`mus`; the placement re-points it from `{corr: a}` onto
`{jes: up, mus: up}`. A re-point needs a genuinely new **≥2-coordinate** point — a single `{jes: up}`
is already the `jes_up` label's point, and Level 18 shows it refused.

In [16]:
s, c = ctx0()
pt = c["pt"]

# two carrier axes a placement can name: jes on the jets, a muon momentum scale (mus) on the muons
carriers = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
mu_pt = carriers["mu_pt"]
carriers = graphed.vary(carriers, "mus", collections={"mu_pt": {"up": mu_pt * 1.01, "down": mu_pt * 0.99}})

# an INDEPENDENT weight: read off graphed.nominal, so it carries neither jes nor mus
factor = graphed.nominal(carriers["pt"]) * 0.5
reg = graphed.vary(
    carriers,
    "corr",
    factor,
    is_weight=True,
    points=[
        ("a", factor * 3.0),  # DECLARE corr_a
        {"corr": "a", "jes": "up", "mus": "up"},
    ],
)  # PLACE it off-grid

show(graphed.weight(reg))
print()
print(
    "corr_a re-pointed to",
    graphed.points(graphed.weight(reg))["corr_a"],
    "-- own 'corr' axis dropped; the label is a name for the point.",
)

labels    : ('nominal', 'jes_up', 'jes_down', 'mus_up', 'mus_down', 'corr_a')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    mus_up                 {'mus': 'up'}
    mus_down               {'mus': 'down'}
    corr_a                 {'jes': 'up', 'mus': 'up'}

corr_a re-pointed to {'jes': 'up', 'mus': 'up'} -- own 'corr' axis dropped; the label is a name for the point.


## Level 16 — the μR×μF 7-point scale set

The textbook off-grid case. The renormalisation and factorisation scales, μR and μF, are
**independent** one-at-a-time families (each doubled and halved), but the accepted envelope is the
**7-point set**: the four axis-aligned variations, the nominal, and the two **correlated diagonals**
`(μR, μF) = (2, 2)` and `(0.5, 0.5)` — never the anti-correlated `(2, 0.5)` corners.

The diagonals are exactly two off-grid placements. A `scale` family declares two independent members
`upup`/`dndn`, then places each at the two-coordinate diagonal point over the μR and μF axes. Numeric tags
and coordinates are values, written as numbers: each canonicalises to one label, `2 → "2"`,
`0.5 → "5em1"` (a filesystem-safe rendering of the value). The next cell walks every spelling.

Two of these registrations re-use a node the ambient already has (Level 6's rule): `muF`'s nominal
is the `graphed.weight()` handle, so `muF` is a **relative delta** over the `muR` factor and `muF_2`
is 1.3 × the nominal weight; and `scale`'s `base` is the same expression as `muR`'s central, hence
the same node, so `scale` **joins** the `muR` factor instead of multiplying a second copy of it in.
Each diagonal is then exactly the member declared for it — a placed point carrying a relative-delta
family's coordinate keeps the value you declared there — so `scale_upup` is `base * 1.5`, with no
further 1.3 on it. Before the rule all three registrations were factors, so the nominal weight
printed below was the μR weight **cubed**; the labels and the points are unchanged.


In [17]:
s, c = ctx0()
pt = c["pt"]

# muR and muF: INDEPENDENT weight families with numeric tags -- the 4 axis-aligned scale variations
mu_r = graphed.vary(c, "muR", pt * 0.5, is_weight=True, points={2: pt * 0.6, 0.5: pt * 0.4})
ambient = graphed.weight(mu_r)
mu_f = graphed.vary(mu_r, "muF", ambient, is_weight=True, points={2: ambient * 1.3, 0.5: ambient * 0.7})

# the two correlated DIAGONALS: declare upup/dndn independent, place each at a (muR, muF) point
base = pt * 0.5
scale = graphed.vary(
    mu_f,
    "scale",
    base,
    is_weight=True,
    points=[
        ("upup", base * 1.5),
        ("dndn", base * 0.87),
        {"scale": "upup", "muR": 2, "muF": 2},
        {"scale": "dndn", "muR": 0.5, "muF": 0.5},
    ],
)

show(graphed.weight(scale))
print()
print(
    len(graphed.labels(graphed.weight(scale))),
    "universes = nominal + 4 axis-aligned (muR/muF) + 2 correlated diagonals (not the (2, 0.5) corners)",
)

labels    : ('nominal', 'muR_2', 'muR_5em1', 'muF_2', 'muF_5em1', 'scale_upup', 'scale_dndn')
points    :
    nominal                {}
    muR_2                  {'muR': '2'}
    muR_5em1               {'muR': '5em1'}
    muF_2                  {'muF': '2'}
    muF_5em1               {'muF': '5em1'}
    scale_upup             {'muF': '2', 'muR': '2'}
    scale_dndn             {'muF': '5em1', 'muR': '5em1'}

7 universes = nominal + 4 axis-aligned (muR/muF) + 2 correlated diagonals (not the (2, 0.5) corners)


### Every spelling of a numeric tag

A numeric tag is a **value**, not a string. Integers, floats, numpy scalars and decimal strings all
canonicalise by exact decimal arithmetic to one label, `m?<digits>(em<digits>)?` — a sign becomes
`m`, a negative exponent `em` — so `2`, `2.0`, `"2"`, `"02"`, `"2.00"` and `np.int64(2)` are one tag
and one universe, and a mapping that spells one value twice is refused. The exact value survives the
rendering: `graphed.variations` reports each tag as a `Fraction`. Placement coordinates take the same
spellings. Refused: booleans, `Fraction`/`Decimal`, non-finite values, and strings the grammar does not
read (`"+2"`, `".5"`) — a label has to be usable as a column or category name. A float is its shortest
round-tripping decimal, so `1/3` becomes a sixteen-digit tag: name such a point with an identifier
instead.


In [18]:
from fractions import Fraction

# ints, floats, numpy scalars and decimal strings: all values, each its own universe
s, c = ctx0()
pt = c["pt"]
sigma = graphed.vary(
    c,
    "jes",
    pt * 0.5,
    is_weight=True,
    points={2.5: pt * 0.6, -2.5: pt * 0.4, np.float32(0.5): pt * 0.55, "1e-3": pt * 0.51},
)
print("labels      ", graphed.labels(graphed.weight(sigma)))
print("exact values", graphed.variations(sigma)["jes"])


def tag_of(key):
    """The label one spelling mints for a `muR` family."""
    _s, c = ctx0()
    pt = c["pt"]
    return graphed.labels(
        graphed.weight(graphed.vary(c, "muR", pt * 0.5, is_weight=True, points={key: pt * 0.6}))
    )[1]


# one value, six spellings, ONE tag
print("one tag     ", {repr(x): tag_of(x) for x in (2, 2.0, "2", "02", "2.00", np.int64(2))})
print("1/3         ", tag_of(1 / 3), " <- shortest round-trip decimal; name such a point instead")

# placement coordinates take the same spellings (a fresh Session: one point wears one label)
s, c = ctx0()
pt = c["pt"]
r = graphed.vary(c, "muR", pt * 0.5, is_weight=True, points={2: pt * 0.6, 0.5: pt * 0.4})
f = graphed.vary(r, "muF", graphed.weight(r), is_weight=True, points={2: pt * 0.65, 0.5: pt * 0.35})
same = graphed.vary(
    f,
    "scale",
    pt * 0.5,
    is_weight=True,
    points=[("upup", pt * 0.75), {"scale": "upup", "muR": "2.0", "muF": 2}],
)
print("coordinates ", graphed.points(graphed.weight(same))["scale_upup"])

# refused, each inside vary with nothing minted
for bad in (
    {2: 0.6, "2.0": 0.7},
    {True: 0.6},
    {Fraction(1, 2): 0.6},
    {float("inf"): 0.6},
    {"+2": 0.6},
    {".5": 0.6},
):
    s, c = ctx0()
    pt = c["pt"]
    try:
        graphed.vary(c, "x", pt * 0.5, is_weight=True, points={k: pt * f for k, f in bad.items()})
    except graphed.GraphedError as e:
        print(f"refused {list(bad)!r:>22} -> {e}")

labels       ('nominal', 'jes_25em1', 'jes_m25em1', 'jes_5em1', 'jes_1em3')
exact values {'25em1': (Kind.WEIGHT, Fraction(5, 2)), 'm25em1': (Kind.WEIGHT, Fraction(-5, 2)), '5em1': (Kind.WEIGHT, Fraction(1, 2)), '1em3': (Kind.WEIGHT, Fraction(1, 1000))}
one tag      {'2': 'muR_2', '2.0': 'muR_2', "'2'": 'muR_2', "'02'": 'muR_2', "'2.00'": 'muR_2', 'np.int64(2)': 'muR_2'}
1/3          muR_3333333333333333em16  <- shortest round-trip decimal; name such a point instead
coordinates  {'muF': '2', 'muR': '2'}
refused             [2, '2.0'] -> tags '2.0' and one already given canonicalize to '2': one value cannot name two universes
refused                 [True] -> variation tags must be strings, integers or floats, got True
refused       [Fraction(1, 2)] -> variation tags must be strings, integers or floats, got Fraction(1, 2)
refused                  [inf] -> variation tag 'inf' does not name a finite value
refused                 ['+2'] -> variation tag '+2' is neither a numeric spelling no

## Level 17 — prescribed directions, and mixing the two verbs in one call

The diagonal of Level 16 is a **prescribed direction**: a universe an analyst names by hand as a point
over foreign axes. The same shape carries a reparameterised or rotated basis — a **PDF Hessian
eigenvector direction**, a decorrelated JES splitting — wherever the wanted universe is a combination
of parameters rather than one axis's own tag. Below, `corr_off` is placed at `{jes: 1, btag: -1}`, a
two-coordinate point over **two different** families; both coordinates are validated by the same
carrier-reachability walk.

Both weight families in the first program are **relative deltas** (Level 6's rule): `btag`'s
nominal is the handle `w1` and `corr`'s is the handle `ambient`, so each replaces the running product
at its own labels rather than multiplying into it. `corr_off` is therefore the member declared for
the prescribed point, `ambient * 1.1` — one 1.1 on the nominal weight, whatever its point says about
`jes` and `btag`. Before that rule the handle became another factor, so this cell's nominal weight
was the `jes` weight to the **fourth** power; the printed points are unchanged.

Declares, prunes, and additive re-points all ride **one** `points=` list and are routed
per-entry, so a single call can prune a *dependent* member's grid **and** re-point an *independent*
member off-grid at once — they act on disjoint members and never contend.

In [19]:
# a prescribed direction over TWO different families: {jes: 1, btag: -1}
s, c = ctx0()
pt = c["pt"]
jes = graphed.vary(c, "jes", pt * 0.5, is_weight=True, points={1: pt * 0.6, -1: pt * 0.4})
w1 = graphed.weight(jes)
btag = graphed.vary(jes, "btag", w1, is_weight=True, points={1: w1 * 1.3, -1: w1 * 0.7})
ambient = graphed.weight(btag)
corr = graphed.vary(
    btag,
    "corr",
    ambient,
    is_weight=True,
    points=[("off", ambient * 1.1), {"corr": "off", "jes": 1, "btag": -1}],
)
print(
    "prescribed direction   corr_off ->",
    graphed.points(graphed.weight(corr))["corr_off"],
    "  (1 -> '1', -1 -> 'm1')",
)

# ONE call: a PRUNE (dependent member) and an ADDITIVE re-point (independent member) together
s, c = ctx0()
pt = c["pt"]
a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
mu_pt = a["mu_pt"]
carriers = graphed.vary(a, "mus", collections={"mu_pt": {"up": mu_pt * 1.01, "down": mu_pt * 0.99}})

dependent = carriers["pt"] * 0.5  # jes-varied -> the corr family fans over jes
independent = graphed.nominal(carriers["pt"]) * 0.7  # off the nominal -> carries no foreign axis
mixed = graphed.vary(
    carriers,
    "corr",
    dependent,
    is_weight=True,
    points=[
        ("dep", dependent * 1.3),
        ("ind", independent * 1.1),
        {"corr": "dep", "jes": "up"},  # PRUNE: keep this joint
        {"corr": "ind", "jes": "up", "mus": "up"},
    ],
)  # ADDITIVE: re-point
mp = graphed.points(graphed.weight(mixed))
print()
print("prune    corr_dep__jes_up ->", mp["corr_dep__jes_up"], " (own 'corr' axis KEPT -- a real cross)")
print("         corr_dep__jes_down dropped:", "corr_dep__jes_down" not in mp)
print("additive corr_ind         ->", mp["corr_ind"], " (own 'corr' axis DROPPED)")

prescribed direction   corr_off -> {'btag': 'm1', 'jes': '1'}   (1 -> '1', -1 -> 'm1')

prune    corr_dep__jes_up -> {'corr': 'dep', 'jes': 'up'}  (own 'corr' axis KEPT -- a real cross)
         corr_dep__jes_down dropped: True
additive corr_ind         -> {'jes': 'up', 'mus': 'up'}  (own 'corr' axis DROPPED)


## Level 18 — the construction-time refusals, discriminated by `.situation`

Every way `points=` can be misused raises **`graphed.PointError`** — a `GraphedError`
subclass whose `.situation` string names which contract broke. So `except graphed.PointError`
(or the broader `except graphed.GraphedError`) catches them all, and `.situation` tells them apart.
The complete set, each fired below:

| `.situation` | the entry that triggers it |
|---|---|
| `unreachable` | a coordinate that is not a registered tag of its axis (`{jes: sideways}`) |
| `conflict` | a placement together with `composes_as_union=True` — the union throws every joint away |
| `empty` | a placement carrying only its own-name coordinate — the nominal universe already is that |
| `duplicate` | a point already named by another label — a single foreign `{jes: up}` is just `jes_up` |
| `unresolved` | a placement whose own tag was never declared, or a nuisance registered nowhere this call sees |

The `max_universes=` budget is a **separate** guard, not a `PointError`: a plain `GraphedError`
with no `.situation`. A default grid past the budget is refused *before* it is minted (Level 14),
naming the count and the families so the fix — a placement selection or a raised budget — is obvious.

In [20]:
# ---- refusals over a DEPENDENT grid (Level 14's sf family, off the jes-varied jets) ----
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate, args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


declares = [("up", sf("up")), ("down", sf("down"))]

# unreachable: 'sideways' is not a registered jes tag -- it names no universe the grid derives
try:
    graphed.vary(sf("central"), "sf", points=[*declares, {"sf": "up", "jes": "sideways"}])
except graphed.PointError as e:
    print(f"unreachable  .situation={e.situation!r}\n   {e}\n")

# conflict: a placement AND composes_as_union -- the union collapses every joint away
try:
    graphed.vary(sf("central"), "sf", points=[*declares, {"sf": "up", "jes": "up"}], composes_as_union=True)
except graphed.PointError as e:
    print(f"conflict     .situation={e.situation!r}\n   {e}\n")


# ---- refusals over an INDEPENDENT member (the additive re-point of Level 15) ----
def repoint(entry):
    """Register an independent corr member on a fresh jes+mus context with one placement entry."""
    _s, c = ctx0()
    pt = c["pt"]
    a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
    ev = a["mu_pt"]
    car = graphed.vary(a, "mus", collections={"mu_pt": {"up": ev * 1.01, "down": ev * 0.99}})
    factor = graphed.nominal(car["pt"]) * 0.5
    return graphed.vary(car, "corr", factor, is_weight=True, points=[("a", factor * 3.0), entry])


# empty: a placement carrying only its own coordinate -- the nominal universe already IS that point
try:
    repoint({"corr": "a"})
except graphed.PointError as e:
    print(f"empty        .situation={e.situation!r}\n   {e}\n")

# duplicate: a single foreign {jes: up} is already the jes_up label's point (a label needs >=2 coords)
try:
    repoint({"corr": "a", "jes": "up"})
except graphed.PointError as e:
    print(f"duplicate    .situation={e.situation!r}\n   {e}\n")

# unresolved: the placement's own tag 'typo' was never declared
try:
    repoint({"corr": "typo", "jes": "up", "mus": "up"})
except graphed.PointError as e:
    print(f"unresolved   .situation={e.situation!r}\n   {e}\n")

# the max_universes budget is a SEPARATE guard -- a plain GraphedError, no .situation
try:
    graphed.vary(sf("central"), "sf", points=dict(declares), max_universes=8)
except graphed.GraphedError as e:
    print(f"over budget  type={type(e).__name__}, has .situation={hasattr(e, 'situation')}\n   {e}")

unreachable  .situation='unreachable'
   unreachable: a placement on graphed.vary('sf'): 'sideways' is not a registered tag of nuisance 'jes', whose tags are ['down', 'up']

conflict     .situation='conflict'
   conflict: graphed.vary('sf') got both composes_as_union=True and a placement; the union collapses every joint away, so there is no joint for a placement to keep

empty        .situation='empty'
   empty: points= entry {'corr': 'a'} has only the 'corr' coordinate; a foreign coordinate at 0 names the central universe, which is what nominal already is

duplicate    .situation='duplicate'
   duplicate: point {'jes': 'up'} is already registered under label 'jes_up', so label 'corr_a' would be a second name for one universe — two slots, two StrCategory bins and two content hashes

unresolved   .situation='unresolved'
   unresolved: points= entry {'corr': 'typo', 'jes': 'up', 'mus': 'up'}: 'typo' is not a tag of graphed.vary('corr'), whose tags are ['a']

over budget  type=GraphedErro

## Level 19 — putting it together: JEC → jets → Type-1 MET → b-tag

The CMS jet-and-MET stack, mocked end to end, so that every mechanism above appears at once on one
context. The prescription is the one coffea, PocketCoffea and the BTV payloads implement:

- **JEC** — one L1L2L3 factor rescales jet `pt` **and** `mass`; on MC the jets are then **JER-smeared**
  with the hybrid method: scaled toward a matched gen jet within 3σ, else smeared with a stored N(0,1)
  draw ([coffea `CorrectedJetsFactory`](https://github.com/scikit-hep/coffea/blob/master/src/coffea/jetmet_tools/CorrectedJetsFactory.py)).
- **JES up/down** — `(1 ± δ)` on `pt` and `mass`, δ evaluated at the smeared `pt`. **JER up/down** —
  the same smearing with the up/down resolution scale factor.
- **Type-1 MET** — from the **raw** MET, minus the momentum change of every jet with corrected
  `pt > 15 GeV` and EM fraction `< 0.9`, the selection frozen at nominal; **recomputed in every JES and
  JER universe** ([coffea `CorrectedMETFactory`](https://github.com/scikit-hep/coffea/blob/master/src/coffea/jetmet_tools/CorrectedMETFactory.py),
  [coffea #1631](https://github.com/scikit-hep/coffea/pull/1631),
  [PocketCoffea `jets.py`](https://github.com/PocketCoffea/PocketCoffea/blob/main/pocket_coffea/lib/jets.py)).
- **Unclustered energy** — `MET ± (ΔX, ΔY)` on the nominal-jet MET; the jets do not move.
- **b-tag shape SF** — the per-event weight is the product of `SF(systematic, flavour, |η|, pt, disc)`
  over the taggable jets. Inside a JES universe the weight is the **`up_jes` SF on the shifted jets**,
  *replacing* the nominal weight — the payload says so verbatim: *"to be correlated with the
  respective jes uncertainty sources (not additional nuisance parameters!)"*. The method sources
  (`hf`, `lf`, `hfstats`, `lfstats`, `cferr`) are independent one-at-a-time weights on nominal jets;
  `cferr` moves c jets only, the others b and light jets
  ([PocketCoffea `sf_btag`](https://github.com/PocketCoffea/PocketCoffea/blob/main/pocket_coffea/lib/scale_factors.py),
  [BTV `btagging.json` description](https://github.com/TopEFT/topcoffea/blob/main/topcoffea/data/POG/BTV/2022_Summer22EE/btagging.json.gz)).

Each coupling maps onto one mechanism from the tour — and only the last row needs a `points=` entry:

| coupling | mechanism | how it is written |
|---|---|---|
| JES/JER move jets **and** MET together | lockstep (level 5) | one `collections={"Jet": …, "MET": …}` call per nuisance |
| unclustered energy moves MET alone | one-at-a-time (level 6) | members built off `graphed.nominal(MET)` |
| the nominal b-tag weight moves with the jets | propagation (level 9) | `btag_weight(ctx["Jet"], "central")` over the *Varied* jets |
| the `up_jes` SF belongs to the `jes` universe | name identity (level 8) | `graphed.vary(ctx, "jes", central, is_weight=True, up=…)` |
| `hf`/`lf`/`cferr` are their own nuisances | stacked families (level 6) | one `is_weight=True` call each, on nominal jets |
| MET × b-tag joints, for the factorization error | auto-fanout (level 10) | evaluate a source over the *Varied* jets |
| keep only chosen joints | placement prune (level 14) | `points=[…, {"btag_hf": "up", "jer": "up"}]` |

The mocks keep the real payloads' inputs and node structure (formula and category nodes) with
made-up numbers, and drop what the mechanism does not need: no muon subtraction, L1 = raw, one JES
source, three of the eight method sources.


In [21]:
def toy_events(n_events=6, seed=19):
    """An MC-like sample with the fields the JetMET and BTV recipes read (NanoAOD names in comments)."""
    rng = np.random.default_rng(seed)
    counts = rng.integers(2, 5, size=n_events)
    nj = int(counts.sum())
    pt_raw = rng.exponential(45.0, nj) + 15.0
    matched = rng.uniform(size=nj) < 0.7  # Jet_genJetIdx >= 0
    jets = ak.zip(
        {
            "pt_raw": pt_raw,
            "mass_raw": 0.1 * pt_raw,  # Jet_pt * (1 - Jet_rawFactor)
            "eta": rng.uniform(-2.4, 2.4, nj),
            "phi": rng.uniform(-np.pi, np.pi, nj),
            "emf": rng.uniform(0.0, 1.2, nj),  # Jet_chEmEF + Jet_neEmEF
            "genpt": ak.mask(
                ak.Array(pt_raw * rng.normal(1.0, 0.08, nj)), matched
            ),  # GenJet_pt via genJetIdx
            "flavour": rng.choice(np.array([0, 4, 5]), nj, p=[0.6, 0.15, 0.25]),  # Jet_hadronFlavour
            "disc": rng.uniform(size=nj),  # Jet_btagDeepFlavB
            "draw": rng.normal(
                size=nj
            ),  # the JER stochastic draw: one N(0,1) per jet, reused by every universe
        }
    )
    return ak.Array(
        {
            "Jet": ak.unflatten(jets, counts),
            "RawMET": ak.zip(
                {"pt": rng.exponential(30.0, n_events) + 5.0, "phi": rng.uniform(-np.pi, np.pi, n_events)}
            ),
            "MET": ak.zip(
                {
                    "unclust_dx": rng.normal(0.0, 4.0, n_events),  # MET_MetUnclustEnUpDeltaX
                    "unclust_dy": rng.normal(0.0, 4.0, n_events),
                }
            ),
            "rho": rng.uniform(10.0, 30.0, n_events),  # fixedGridRhoFastjetAll
        }
    )


def formula(expr, *variables):
    return {"nodetype": "formula", "expression": expr, "parser": "TFormula", "variables": list(variables)}


def category(input_name, content):
    return {
        "nodetype": "category",
        "input": input_name,
        "content": [{"key": k, "value": v} for k, v in content.items()],
    }


def correction(name, inputs, data):
    return {
        "name": name,
        "version": 1,
        "output": {"name": "out", "type": "real"},
        "data": data,
        "inputs": [{"name": n, "type": t} for n, t in inputs],
    }


# (b, c, light) multipliers on the central shape SF. The POG file's flavour rule: hf/lf/jes act on
# b and light jets, c jets move only through cferr -- so the c column is 1.0 everywhere else.
BTAG_K = {
    "central": (1.0, 1.0, 1.0),
    "up_jes": (1.03, 1.0, 1.02),
    "down_jes": (0.97, 1.0, 0.98),
    "up_hf": (1.06, 1.0, 1.01),
    "down_hf": (0.94, 1.0, 0.99),
    "up_lf": (1.01, 1.0, 1.08),
    "down_lf": (0.99, 1.0, 0.92),
    "up_cferr1": (1.0, 1.10, 1.0),
    "down_cferr1": (1.0, 0.90, 1.0),
}

# mock-ups of the real payloads: the real inputs and node structure, made-up numbers
MOCK = json.dumps(
    {
        "schema_version": 2,
        "corrections": [
            correction(
                "jec",
                [("pt", "real"), ("eta", "real")],  # L1L2L3 as ONE factor
                formula("1.12 - 0.0004*x + 0.01*abs(y)", "pt", "eta"),
            ),
            correction(
                "jes_unc",
                [("eta", "real"), ("pt", "real")],  # Total JES uncertainty
                formula("0.01 + 0.01*abs(x) + 1.0/y", "eta", "pt"),
            ),
            correction(
                "jer_res",
                [("pt", "real"), ("rho", "real")],  # relative pT resolution
                formula("(0.8 + 0.02*y)/sqrt(x) + 0.02", "pt", "rho"),
            ),
            correction(
                "jer_sf",
                [("systematic", "string"), ("eta", "real")],  # JER scale factor
                category(
                    "systematic",
                    {
                        "nom": formula("1.10 + 0.05*abs(x)", "eta"),
                        "up": formula("1.20 + 0.08*abs(x)", "eta"),
                        "down": formula("1.02 + 0.02*abs(x)", "eta"),
                    },
                ),
            ),
            correction(
                "btag_shape",
                [
                    ("systematic", "string"),
                    ("flavour", "int"),
                    ("abseta", "real"),
                    ("pt", "real"),
                    ("discriminant", "real"),
                ],  # deepJet shape SF
                category(
                    "systematic",
                    {
                        syst: category(
                            "flavour",
                            {
                                5: formula(f"{kb}*(0.95 + 0.0005*x + 0.05*y)", "pt", "discriminant"),
                                4: formula(f"{kc}*(1.00 + 0.0003*x)", "pt", "discriminant"),
                                0: formula(f"{kl}*(1.02 - 0.0002*x + 0.10*y)", "pt", "discriminant"),
                            },
                        )
                        for syst, (kb, kc, kl) in BTAG_K.items()
                    },
                ),
            ),
        ],
    }
).encode()
CSET = correctionlib.CorrectionSet.from_string(MOCK.decode())


def corr(name, inputs, args):
    """One mock correction as an External node (level 9): it maps over Varied inputs."""
    return gak.apply_correction(MOCK, name, inputs, CSET[name].evaluate, args=args)


def rescale(jets, factor):
    """A jet-energy factor moves pt AND mass together (level 5) and nothing else."""
    return gak.with_field(gak.with_field(jets, jets.pt * factor, "pt"), jets.mass * factor, "mass")


def apply_jec(raw):
    jets = gak.with_field(gak.with_field(raw, raw.pt_raw, "pt"), raw.mass_raw, "mass")
    return rescale(jets, corr("jec", [jets.pt, jets.eta], ["$0", "$1"]))


def jer_factor(jets, rho, systematic):
    """Hybrid JER: scale to the matched gen jet within 3 sigma, else smear with the stored draw."""
    sigma = corr("jer_res", [jets.pt, jets.pt * 0.0 + rho], ["$0", "$1"])
    sf = corr("jer_sf", [jets.eta], [systematic, "$0"])
    genpt = gak.fill_none(jets.genpt, 0.0)
    matched = ~gak.is_none(jets.genpt, axis=1) & (np.abs(jets.pt - genpt) < 3.0 * sigma * jets.pt)
    scaling = 1.0 + (sf - 1.0) * (jets.pt - genpt) / jets.pt
    smearing = 1.0 + jets.draw * sigma * np.sqrt(np.maximum(sf * sf - 1.0, 0.0))
    floor = 0.01 / np.cosh(jets.eta) / jets.pt  # coffea's minimum jet energy, 10 MeV
    return np.maximum(gak.where(matched, scaling, smearing), floor)


def type1_met(raw_met, raw, jets, selected):
    """Type-1: RAW MET minus the momentum change of the selected jets (mock: L1 = raw, no muons)."""
    dpt = (jets.pt - raw.pt_raw)[selected]
    dpx = gak.sum(dpt * np.cos(raw.phi[selected]), axis=1)
    dpy = gak.sum(dpt * np.sin(raw.phi[selected]), axis=1)
    px = raw_met.pt * np.cos(raw_met.phi) - dpx
    py = raw_met.pt * np.sin(raw_met.phi) - dpy
    return gak.zip({"pt": np.hypot(px, py), "phi": np.arctan2(py, px)})


def btag_weight(jets, systematic):
    """Product of the per-jet shape SF over the taggable jets -- a cut on the (possibly shifted) pT."""
    tagged = jets[(jets.pt > 30.0) & (abs(jets.eta) < 2.4)]
    sf = corr(
        "btag_shape",
        [tagged.flavour, abs(tagged.eta), tagged.pt, tagged.disc],
        [systematic, "$0", "$1", "$2", "$3"],
    )
    return gak.prod(sf, axis=1)


def moved(ctx, labels=None):
    """One row per universe: event 0's leading-jet pT, MET pT and weight, then the point.

    `graphed.universe(ctx, label)` is a CHILD context with every collection and the ambient weight
    resolved at that label by projection (level 11), so an axis a collection does not carry reads
    as nominal.
    """
    pts = graphed.points(ctx)
    print(f"    {'label':24s} {'jet0 pt':>8s} {'MET pt':>8s} {'weight':>8s}   point")
    for label in labels or graphed.labels(ctx):
        u = graphed.universe(ctx, label)
        w = graphed.weight(u)
        print(
            f"    {label:24s} {float(s.materialize(u['Jet']).pt[0, 0]):8.2f}"
            f" {float(s.materialize(u['MET']).pt[0]):8.2f}"
            f" {float(s.materialize(w)[0]) if w is not None else 1.0:8.4f}   {pts[label]}"
        )


print("mock corrections:", list(CSET))

mock corrections: ['btag_shape', 'jec', 'jer_res', 'jer_sf', 'jes_unc']


### The nominal chain

Raw jets → JEC → JER smearing = the nominal jets; the Type-1 MET follows from them. Nothing is
varied yet, so the context has one universe.


In [22]:
s = Session(AwkwardBackend())
ev = from_awkward(s, "ev", toy_events())
raw, raw_met, rho = ev.Jet, ev.RawMET, ev.rho

jec = apply_jec(raw)  # L1L2L3-corrected jets
nominal_jets = rescale(jec, jer_factor(jec, rho, "nom"))  # + nominal JER smearing = the nominal jets
in_type1 = (nominal_jets.pt > 15.0) & (raw.emf < 0.9)  # the Type-1 selection, frozen at NOMINAL
c = EventContext(
    s, ev, collections={"Jet": nominal_jets, "MET": type1_met(raw_met, raw, nominal_jets, in_type1)}
)

print("event 0   raw jet pt :", [round(x, 2) for x in ak.to_list(s.materialize(raw.pt_raw)[0])])
print("      corrected jet pt:", [round(x, 2) for x in ak.to_list(s.materialize(c["Jet"]).pt[0])])
print(
    "      raw -> Type-1 MET:",
    round(float(s.materialize(raw_met).pt[0]), 2),
    "->",
    round(float(s.materialize(c["MET"]).pt[0]), 2),
)

event 0   raw jet pt : [18.84, 56.89, 95.37]
      corrected jet pt: [21.27, 71.48, 105.43]
      raw -> Type-1 MET: 16.65 -> 27.82


### Shifts: JES and JER move jets and MET in lockstep; unclustered energy moves MET alone

The `jes` jets are varied with the loose form on the context's central jets, and the Type-1 MET is
recomputed **from that container**, so it is `Varied` over `jes` by propagation; the shift form takes
both containers as they are — level 5's lockstep form without spelling the tag set twice — and
`jes_up` is a single universe in which both moved. A container is accepted there only when it carries
exactly the family being registered on the context's own collection, which is why the JER container
is built on the nominal jets rather than on `c["Jet"]` (by then `Varied` over `jes`): `jer` is
independent of `jes` and the two compose as the union. The unclustered members are built off `graphed.nominal(c["MET"])`: they carry
no foreign axis, so they are one-at-a-time by construction.

Every row of the table is one universe; read which columns moved.


In [23]:
jet = c["Jet"]
delta = corr("jes_unc", [jet.eta, jet.pt], ["$0", "$1"])  # evaluated at the smeared pT
jets = graphed.vary(jet, "jes", up=rescale(jet, 1.0 + delta), down=rescale(jet, 1.0 - delta))
c = graphed.vary(c, "jes", collections={"Jet": jets, "MET": type1_met(raw_met, raw, jets, in_type1)})
jets = graphed.vary(
    nominal_jets,
    "jer",
    up=rescale(jec, jer_factor(jec, rho, "up")),
    down=rescale(jec, jer_factor(jec, rho, "down")),
)
c = graphed.vary(c, "jer", collections={"Jet": jets, "MET": type1_met(raw_met, raw, jets, in_type1)})

met0 = graphed.nominal(c["MET"])  # unclustered energy: NOMINAL jets
px, py = met0.pt * np.cos(met0.phi), met0.pt * np.sin(met0.phi)
dx, dy = ev.MET.unclust_dx, ev.MET.unclust_dy
c = graphed.vary(
    c,
    "unclustered",
    collections={
        "MET": {
            "up": gak.zip({"pt": np.hypot(px + dx, py + dy), "phi": np.arctan2(py + dy, px + dx)}),
            "down": gak.zip({"pt": np.hypot(px - dx, py - dy), "phi": np.arctan2(py - dy, px - dx)}),
        }
    },
)

print("variations:", {k: {t: kind for t, (kind, _) in v.items()} for k, v in graphed.variations(c).items()})
moved(c)

variations: {'jes': {'up': Kind.SHIFT, 'down': Kind.SHIFT}, 'jer': {'up': Kind.SHIFT, 'down': Kind.SHIFT}, 'unclustered': {'up': Kind.SHIFT, 'down': Kind.SHIFT}}
    label                     jet0 pt   MET pt   weight   point
    nominal                     21.27    27.82   1.0000   {}
    jes_up                      22.55    29.42   1.0000   {'jes': 'up'}
    jes_down                    19.99    26.23   1.0000   {'jes': 'down'}
    jer_up                      21.51    30.63   1.0000   {'jer': 'up'}
    jer_down                    21.07    24.36   1.0000   {'jer': 'down'}
    unclustered_up              21.27    29.06   1.0000   {'unclustered': 'up'}
    unclustered_down            21.27    28.73   1.0000   {'unclustered': 'down'}


### The b-tag weight: propagation, name identity, and the method sources

Three registrations, three mechanisms, no `points=`:

1. `central` is computed over `c["Jet"]`, which is **Varied** over `jes` and `jer` — so inside `jes_up`
   the nominal SF is re-evaluated on the shifted jets. That is propagation; nothing declares it.
2. The `up_jes`/`down_jes` SFs are **not a nuisance**. Registering them under the name `jes` with
   `is_weight=True` ties them to the existing shift (level 8): `graphed.variations()` reports `jes`
   as `Kind.WEIGHT|SHIFT`, the label set does not grow, and the `jes_up` universe now moves jets, MET and weight.
   The members are read at `graphed.member_of(jet, "jes_up")` — the JES-shifted jets only — so the
   registration depends on no other axis.
3. The method sources are independent families on the **nominal** jets, each entering as its
   **ratio to central** over a unit nominal, so its universe rescales the central SF the `jes`
   factor already carries instead of replacing it. All three pass the *same* unit-nominal node, so
   under Level 6's rule the second and third **join** the first's factor: `graphed.explain` reports
   the three as three values of one operation, which is also why no `btag_hf × btag_lf` joint
   exists. Naming the SF central itself would join the `jes` factor and take absolute SFs as
   members — the ratio spelling is what keeps these sources on an operation of their own.


In [24]:
jet = c["Jet"]  # Varied over jes and jer; unclustered never touches jets
central = btag_weight(jet, "central")  # PROPAGATION: re-evaluated on each universe's own jets
c = graphed.vary(
    c,
    "jes",
    central,
    is_weight=True,  # NAME IDENTITY: the jes-varied SF rides `jes`
    up=btag_weight(graphed.member_of(jet, "jes_up"), "up_jes"),
    down=btag_weight(graphed.member_of(jet, "jes_down"), "down_jes"),
)
print(
    "jes is now",
    {t: kind for t, (kind, _) in graphed.variations(c)["jes"].items()},
    "-- one fit parameter moving jets, MET and the SF",
)
after_jes = c


def ratio(systematic, jets):
    """The ambient weight is a PRODUCT of factors, so a second family on the same SF is its ratio."""
    return btag_weight(jets, systematic) / btag_weight(jets, "central")


one = graphed.nominal(central) * 0.0 + 1.0
for source in ("hf", "lf", "cferr1"):  # the method sources: independent, one-at-a-time, NOMINAL jets
    c = graphed.vary(
        c,
        f"btag_{source}",
        one,
        is_weight=True,
        up=ratio(f"up_{source}", nominal_jets),
        down=ratio(f"down_{source}", nominal_jets),
    )
print(len(graphed.labels(c)), "universes -- the datacard set")
moved(c)

jes is now {'up': Kind.WEIGHT|SHIFT, 'down': Kind.WEIGHT|SHIFT} -- one fit parameter moving jets, MET and the SF
13 universes -- the datacard set
    label                     jet0 pt   MET pt   weight   point
    nominal                     21.27    27.82   1.0277   {}
    jes_up                      22.55    29.42   1.0939   {'jes': 'up'}
    jes_down                    19.99    26.23   0.9637   {'jes': 'down'}
    jer_up                      21.51    30.63   1.0296   {'jer': 'up'}
    jer_down                    21.07    24.36   1.0255   {'jer': 'down'}
    unclustered_up              21.27    29.06   1.0277   {'unclustered': 'up'}
    unclustered_down            21.27    28.73   1.0277   {'unclustered': 'down'}
    btag_hf_up                  21.27    27.82   1.1547   {'btag_hf': 'up'}
    btag_hf_down                21.27    27.82   0.9081   {'btag_hf': 'down'}
    btag_lf_up                  21.27    27.82   1.0483   {'btag_lf': 'up'}
    btag_lf_down                21.27    27.8

### Beyond the prescription: the MET × b-tag joint grid, kept straight

Evaluate `hf` over the **Varied** jets instead and the family depends on both `jes` and `jer`, so
`graphed` mints the `btag_hf × jes` and the `btag_hf × jer` joints (level 10) — the union of the two
cross products, not their product: universes in which the jets and the MET are shifted and the
hf-shifted SF is evaluated on those jets. `jes` is also a *weight* family on this ambient (name
identity, level 8), and what decides fan-out is not the nuisance's kind but what the member was
computed from: hf's members reach `jes` through the shifted jets, so the coordinate is a
dependency and mints its joints; a member computed from the ambient weight itself would reach
`jes` through the composed factor instead, and be read **two-level** — the composed weight at that
universe, rescaled — so it would carry the jets' shift only through the composition and add no
dependence of its own. (Before Level 6's rule such a member also multiplied the whole weight in a
second time; now it replaces it.) In `btag_hf_up__jes_up` the weight is the two-level product, `SF(up_jes)`
times `ratio(up_hf)`, both on the JES-up jets. The checks read each universe back and compare it
with the quantity the prescription says it must equal — including one negative control.


In [25]:
grid = graphed.vary(
    after_jes,
    "btag_hf",
    one,
    is_weight=True,  # hf over the VARIED jets instead
    up=ratio("up_hf", jet),
    down=ratio("down_hf", jet),
)
joints = [L for L in graphed.labels(grid) if "__" in L]
print("hf over the varied jets:", len(graphed.labels(grid)), "universes; the joints:")
moved(grid, labels=joints)


def same(a, b):
    return bool(ak.all(ak.isclose(s.materialize(a), s.materialize(b))))


jer_up_jets, jes_up_jets = graphed.member_of(jet, "jer_up"), graphed.member_of(jet, "jes_up")
u = graphed.universe(grid, "btag_hf_up__jer_up")
uj = graphed.universe(grid, "btag_hf_up__jes_up")
checks = {
    "joint weight  = SF(up_hf) on the JER-up jets": same(
        graphed.weight(u), btag_weight(jer_up_jets, "up_hf")
    ),
    "joint MET     = the JER-up Type-1 MET (projection)": same(
        u["MET"].pt, graphed.member_of(c["MET"], "jer_up").pt
    ),
    "joint weight  is NOT the nominal-jet SF(up_hf)": not same(
        graphed.weight(u), btag_weight(nominal_jets, "up_hf")
    ),
    "jes joint wgt = SF(up_jes) x ratio(up_hf), both on the JES-up jets (two-level)": same(
        graphed.weight(uj), btag_weight(jes_up_jets, "up_jes") * ratio("up_hf", jes_up_jets)
    ),
    "jes joint MET = the JES-up Type-1 MET (projection)": same(
        uj["MET"].pt, graphed.member_of(c["MET"], "jes_up").pt
    ),
    "jes_up weight = SF(up_jes) on the JES-up jets (name identity)": same(
        graphed.weight(graphed.universe(c, "jes_up")), btag_weight(jes_up_jets, "up_jes")
    ),
    "jes_up MET    = Type-1 MET of the JES-up jets (lockstep)": same(
        graphed.universe(c, "jes_up")["MET"].pt, type1_met(raw_met, raw, jes_up_jets, in_type1).pt
    ),
    "unclustered_up: jets and weight stay nominal": same(
        graphed.universe(c, "unclustered_up")["Jet"].pt, nominal_jets.pt
    )
    and same(graphed.weight(graphed.universe(c, "unclustered_up")), graphed.nominal(graphed.weight(c))),
    "btag_lf_up    : jets and MET stay nominal, weight = SF(up_lf) on nominal jets": same(
        graphed.universe(c, "btag_lf_up")["MET"].pt, graphed.nominal(c["MET"]).pt
    )
    and same(graphed.weight(graphed.universe(c, "btag_lf_up")), btag_weight(nominal_jets, "up_lf")),
}
print()
for text, ok in checks.items():
    print(f"    {'OK ' if ok else 'BAD'}  {text}")
assert all(checks.values())

hf over the varied jets: 17 universes; the joints:
    label                     jet0 pt   MET pt   weight   point
    btag_hf_up__jes_up          22.55    29.42   1.2291   {'btag_hf': 'up', 'jes': 'up'}
    btag_hf_up__jes_down        19.99    26.23   1.0829   {'btag_hf': 'up', 'jes': 'down'}
    btag_hf_up__jer_up          21.51    30.63   1.1569   {'btag_hf': 'up', 'jer': 'up'}
    btag_hf_up__jer_down        21.07    24.36   1.1522   {'btag_hf': 'up', 'jer': 'down'}
    btag_hf_down__jes_up        22.55    29.42   0.9665   {'btag_hf': 'down', 'jes': 'up'}
    btag_hf_down__jes_down      19.99    26.23   0.8516   {'btag_hf': 'down', 'jes': 'down'}
    btag_hf_down__jer_up        21.51    30.63   0.9098   {'btag_hf': 'down', 'jer': 'up'}
    btag_hf_down__jer_down      21.07    24.36   0.9061   {'btag_hf': 'down', 'jer': 'down'}

    OK   joint weight  = SF(up_hf) on the JER-up jets
    OK   joint MET     = the JER-up Type-1 MET (projection)
    OK   joint weight  is NOT the nominal-

### The two knobs on the grid — and where `points=` entered

`composes_as_union=True` collapses the joints back to the datacard set; a placement in `points=`
keeps a chosen subset. That placement is the **only** `points=` entry in the whole stack, and it is
an analysis choice, not part of the prescription: every physics coupling above was expressed by
*what a member was computed from* and *which name it was registered under*.


In [26]:
union = graphed.vary(
    after_jes,
    "btag_hf",
    one,
    is_weight=True,
    composes_as_union=True,
    up=ratio("up_hf", jet),
    down=ratio("down_hf", jet),
)
diag = graphed.vary(
    after_jes,
    "btag_hf",
    one,
    is_weight=True,
    points=[
        ("up", ratio("up_hf", jet)),
        ("down", ratio("down_hf", jet)),
        {"btag_hf": "up", "jer": "up"},
        {"btag_hf": "down", "jer": "down"},
        {"btag_hf": "up", "jes": "up"},
    ],
)
print("full grid              :", len(graphed.labels(grid)), "universes,", len(joints), "joints")
print(
    "composes_as_union=True :",
    len(graphed.labels(union)),
    "universes,",
    len([L for L in graphed.labels(union) if "__" in L]),
    "joints -- the datacard set again",
)
print(
    "placement prune        :",
    len(graphed.labels(diag)),
    "universes, joints kept:",
    [L for L in graphed.labels(diag) if "__" in L],
)

full grid              : 17 universes, 8 joints
composes_as_union=True : 9 universes, 0 joints -- the datacard set again
placement prune        : 12 universes, joints kept: ['btag_hf_up__jer_up', 'btag_hf_down__jer_down', 'btag_hf_up__jes_up']


## Level 20 — seeing what you built: `graphed.explain` and the riders

Every level so far printed `labels()` / `points()` / `variations()` — *what* universes exist.
`graphed.explain(ctx)` answers the other question: **how** the calls you wrote became those
universes. It is a pure reader — it re-decides nothing, mints no node a `graphed.weight()` read
would not, and its text carries no node id — so it is safe to call anywhere, including in the
middle of a notebook you are debugging.

Three sections, one line per item:

- **families**, in registration order: the name, the `Kind`, the tags, the context that registered
  it (the cut and projection links from the root) and **how it entered the ambient weight** —
  `a new factor`, `joins the factor carrying …` (Level 6's rule: the nominal named a weight already
  registered), `an overlay over …` (the nominal was the composition itself), or `shifts …` for the
  collection form — followed by every universe the family *placed* and its point.
- **ambient operations**, in the order the composition applies them: the families each operation
  carries, the row-space links it was carried through, and a `fixed at …` mark inside a
  relative-delta family's own universe. A `factor` multiplies its member at each label; an
  `overlay` replaces the running product at the labels its family covers.
- **universes here**: every label this context carries, with where it came from — one at a time, a
  fan-out joint over the dependency it read, a placement, or a relative-delta family's own universe.

Per weight family the line also carries the relations that explain what is *not* there: the families
it **composes with** (they share no registered point, so their joint is a product that needs no
universe of its own — this is why `hf_up__mu_up` is not in the list), the families it **shares the
factor with** (two values of one weight: their joint is absent for the opposite reason), the shifts
it **fans out over** (it read those shifted objects) and the shifts it is **independent of**.

The chain below is this tour in one context: a pile-up factor, the b-tag heavy-flavour source on one
SF, a muon weight declared as a relative delta over the composition read so far, and the
light-flavour source of the **same** SF, placed at a prescribed correlated point. The oracle line is
the nominal weight built by hand, so the printed nominal proves the SF is in the product once.

In [27]:
s, c = ctx0()
pt = c["pt"]
sf = pt * 0.5  # ONE b-tag scale factor, read by both flavour sources
pu = pt * 0.0 + 0.9  # a flat pile-up weight

ctx = graphed.vary(c, "pu", pu, is_weight=True, up=pu * 1.1)  # a NEW factor
ctx = graphed.vary(ctx, "hf", sf, is_weight=True, up=sf * 1.1)  # another new factor
w = graphed.weight(ctx)  # the composition so far: pu * SF
ctx = graphed.vary(ctx, "mu", w, is_weight=True, up=w * 1.05)  # nominal IS w  -> a relative delta
ctx = graphed.vary(
    ctx,
    "lf",
    sf,
    is_weight=True,  # nominal IS sf -> joins hf's factor
    points=[("up", sf * 1.2), {"lf": "up", "hf": "up", "mu": "up"}],
)
print(graphed.explain(ctx))

amb = graphed.weight(ctx)
print()
for label in graphed.labels(amb):
    print(
        f"  {label:8s}", [round(float(x), 4) for x in list(s.materialize(graphed.universe(amb, label)))[:3]]
    )
print(
    "  oracle  ",
    [round(float(x), 4) for x in list(s.materialize(pu * sf))[:3]],
    " <- pu * SF, the scale factor in the product ONCE",
)

graphed.explain: 4 registrations, 3 ambient operations, 5 universes
families (registration order)
  pu (WEIGHT) ['up'] at the root: a new factor; composes with hf, lf, mu
  hf (WEIGHT) ['up'] at the root: a new factor; shares the factor with lf; composes with pu
  mu (WEIGHT) ['up'] at the root: an overlay over hf, pu; composes with lf, pu
  lf (WEIGHT) ['up'] at the root: joins the factor carrying hf, placing a universe at {hf: up, mu: up}; shares the factor with hf; composes with mu, pu
ambient operations (in composition order)
  #0 factor: pu['up'] registered here
  #1 factor: hf['up'], lf['up'] registered here
  #2 overlay: mu['up'] registered here
universes here
  nominal: a point over no registered family
  pu_up: pu, one at a time
  hf_up: hf, one at a time
  mu_up: mu's own universe, a relative-delta family
  lf_up: lf placed at {hf: up, mu: up}

  nominal  [0.45, 0.9, 1.35]
  pu_up    [0.495, 0.99, 1.485]
  hf_up    [0.495, 0.99, 1.485]
  mu_up    [0.4725, 0.945, 1.4175]
  lf_

### Debugging with it

Two mistakes that print a perfectly ordinary label set and a quietly wrong weight. `explain` names
each in one line.

1. **A weight registered before the shift of the objects its central read.** The central was
   evaluated on the nominal jets, so the family carries the pre-shift value and no `btag × jes`
   joint is ever minted. `explain` reports `reads objects later shifted by jes` — neither *fans out
   over* nor *independent of*, the two answers that would be honest — and names the order. The same
   two calls in the other order report `fans out over jes` and mint the joint.
2. **A central that is the whole ambient by accident.** Pass the `graphed.weight()` handle where the
   scale factor was meant and the family becomes a relative delta: its members *replace* the running
   product, so every other factor — here the pile-up weight — silently drops out of that universe.
   The label set is identical to the correct spelling's and only the values differ, which is exactly
   the kind of bug that survives a review; `explain` reports `an overlay over hf, pu` where the
   intended spelling reports `joins the factor carrying hf`.

In [28]:
# 1. the weight registered BEFORE the shift of the jets its central read
s, c = ctx0()
pt = c["pt"]
flat = pt * 0.5  # read off the NOMINAL pt
early = graphed.vary(c, "btag", flat, is_weight=True, up=flat * 1.1)
late = graphed.vary(early, "jes", collections={"pt": {"up": pt * 1.05}})  # the shift arrives AFTER
print("wrong order:", graphed.explain(late).families["btag"])
print("            ", graphed.labels(graphed.weight(late)), "<- no btag x jes joint")
print()

s, c = ctx0()
pt = c["pt"]
shift = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.05}})
on_shifted = shift["pt"] * 0.5  # read off the SHIFTED pt
right = graphed.vary(shift, "btag", on_shifted, is_weight=True, up=on_shifted * 1.1)
print("right order:", graphed.explain(right).families["btag"])
print("            ", graphed.labels(graphed.weight(right)))
print()

# 2. the central that is the whole ambient by accident
s2, c2 = ctx0()
pt2 = c2["pt"]
sf2, pu2 = pt2 * 0.5, pt2 * 0.0 + 0.9
base = graphed.vary(c2, "pu", pu2, is_weight=True, up=pu2 * 1.1)
base = graphed.vary(base, "hf", sf2, is_weight=True, up=sf2 * 1.1)
handle = graphed.weight(base)  # the whole ambient weight, pu * SF

for tag, nominal in (("oops ", handle), ("meant", sf2)):
    lf = graphed.vary(base, "lf", nominal, is_weight=True, up=sf2 * 1.2)  # members: the lf-shifted SF
    amb2 = graphed.weight(lf)
    print(f"{tag}:", graphed.explain(lf).families["lf"])
    print(
        "       ",
        graphed.labels(amb2),
        " lf_up",
        [round(float(x), 4) for x in list(s2.materialize(graphed.universe(amb2, "lf_up")))[:3]],
        " (nominal",
        [round(float(x), 4) for x in list(s2.materialize(graphed.universe(amb2, "nominal")))[:3]],
        ")",
    )

wrong order: btag (WEIGHT) ['up'] at the root: a new factor; reads objects later shifted by jes
             ('nominal', 'btag_up') <- no btag x jes joint

right order: btag (WEIGHT) ['up'] at the root: a new factor; fans out over jes
             ('nominal', 'jes_up', 'btag_up', 'btag_up__jes_up')

oops : lf (WEIGHT) ['up'] at the root: an overlay over hf, pu; composes with hf, pu
        ('nominal', 'pu_up', 'hf_up', 'lf_up')  lf_up [0.6, 1.2, 1.8]  (nominal [0.45, 0.9, 1.35] )
meant: lf (WEIGHT) ['up'] at the root: joins the factor carrying hf; shares the factor with hf; composes with pu
        ('nominal', 'pu_up', 'hf_up', 'lf_up')  lf_up [0.54, 1.08, 1.62]  (nominal [0.45, 0.9, 1.35] )


### The riders under a cut and a projection

Each operation carries a **rider**: its kind, the families it carries, and the row-space links it
came through. `graphed.context.ambient_entries(ctx)` hands them out — the mechanism view that
`explain`'s middle section is built from, and the thing to read when a weight is wrong below a cut.
Under a cut the child adopts one composed weight, so the factors collapse into the single product
its node is, with the overlay behind it keeping its own line. Inside `graphed.universe(ctx,
"hf_up")` the relative-delta operation is **gone**: that universe carries none of `mu`'s coordinate,
so the weight there has no 1.05 in it. Inside `mu_up` — `mu`'s own universe — the overlay is
**fixed**, because every value there *is* that universe, and both the rider and `explain`'s line
say so.

A projection also restricts what may be registered on it. Naming the factor that the projected
universe is a universe *of* would put the central back where the variation belongs, so it is
refused before anything is minted, and the message names the two spellings that work.

In [29]:
from graphed.context import ambient_entries  # the per-operation view, one level below explain

cut = ctx[ctx["pt"] > 3.0]  # a cut: the child adopts ONE composed weight
for tag, child in (
    ("after a cut  ", cut),
    ("inside hf_up ", graphed.universe(ctx, "hf_up")),
    ("inside mu_up ", graphed.universe(ctx, "mu_up")),
):
    print(tag)
    for _slot, rider, _entry in ambient_entries(child):
        fixed = f"  FIXED at {rider.fixed_at}" if rider.fixed else ""
        print(f"   {rider.kind:7s} {dict(rider.families)}  via {rider.links}{fixed}")

# naming the factor that the projected universe is a universe OF
try:
    graphed.vary(graphed.universe(ctx, "hf_up"), "cferr", sf, is_weight=True, up=sf * 1.3)
except graphed.GraphedError as e:
    print(f"\nrefused: {e}")

print("\nand the overlay seen from inside its own universe:")
print(graphed.explain(graphed.universe(ctx, "mu_up")))

after a cut  
   factor  {'pu': ('up',), 'hf': ('up',), 'lf': ('up',)}  via (('mask', None),)
   overlay {'mu': ('up',)}  via (('mask', None),)
inside hf_up 
   factor  {'pu': ('up',), 'hf': ('up',), 'lf': ('up',)}  via (('project', 'hf_up'),)
inside mu_up 
   factor  {'pu': ('up',), 'hf': ('up',), 'lf': ('up',)}  via (('project', 'mu_up'),)
   overlay {'mu': ('up',)}  via (('project', 'mu_up'),)  FIXED at mu_up

refused: graphed.vary('cferr'): its central names the weight factor that the universe 'hf_up' this context is projected into is OF, whose member there is that universe rather than the central; register this family on the factor at the parent, before the projection, or read a graphed.weight() handle AT this context (`w = graphed.weight(ctx)`) and register this family on that

and the overlay seen from inside its own universe:
graphed.explain: 4 registrations, 2 ambient operations, 1 universes
families (registration order)
  pu (WEIGHT) ['up'] at the root: a new factor; composes

## Where each level lands

| # | Level | What it adds |
|---|---|---|
| 1 | loose up/down | `f"{name}_{tag}"`, default point `{name: tag}` |
| 2 | extending a family | a second call, same nuisance — still one fit parameter |
| 3 | weight (`is_weight=True`) | kind `Kind.WEIGHT`; selection fixed |
| 4 | shift (`collections=`) | kind `Kind.SHIFT`; selection moves |
| 5 | lockstep | one nuisance, several collections, no extra universes |
| 6 | stacked families | independent families compose as the **union**: 2+2 → 5, not 9 |
| 7 | shift then weight | the ambient weight carries inherited labels |
| 8 | **name identity** | one name, both effects, one universe; kind `Kind.WEIGHT|SHIFT` |
| 9 | **propagation** | `gak.apply_correction` over a `Varied`, one node per universe |
| 10 | **auto-fanout** | a dependent variation mints the full joint grid — no placement |
| 11 | projection | how a joint label resolves on a container that knows one axis |
| 12 | factorization error | what a joint universe measures, with a zero control |
| 13 | a ROOT file + process pool | the grid crosses the pool; `composes_as_union` is the datacard control |
| 14 | controlling the grid | `composes_as_union` / placement prune / `max_universes` guard |
| 15 | **off-grid placement** | a placement over an *independent* member re-points it off the grid; own axis dropped |
| 16 | μR×μF 7-point set | four axis-aligned + two correlated diagonals, minted as placements |
| 17 | prescribed directions + mixing | a reparameterised direction over two families; prune and re-point in one call |
| 18 | refusals | `PointError.situation`: unreachable / conflict / empty / duplicate / unresolved (+ the `max_universes` guard) |
| 19 | the full stack | JEC → JES/JER lockstep with Type-1 MET, unclustered, b-tag by propagation + name identity; joints kept straight; one optional prune |
| 20 | **`explain` + the riders** | how each family entered the ambient, what the operations are, where every universe came from — and the two tells that catch a wrong weight |

## Traps worth carrying away

- **A dependency mints the grid; independence does not.** A variation computed over another
  nuisance's varied nodes auto-fans (level 10); two independent axes compose as the union (level 6).
  If you expected joints and got a union, the member was not actually built over the varied nodes.
- **`composes_as_union=True` is an analysis choice, applied explicitly.** The default is the full grid, with no analysis choice imposed. A datacard's one-at-a-time set is one keyword away.
- **A placement's meaning follows the member's dependence.** A `{nuisance: coordinate}` map in
  `points=` *prunes* a dependent member's auto-grid (own axis kept, coordinates on-grid) and
  *re-points* an independent member off the grid (own axis dropped, any reachable coordinate). The
  structure of the entry — tuple versus map — picks declare versus place; there is no separate
  placement parameter.
- **`graphed.points()` is the ground truth.** It reports the coordinate map behind every label;
  read it before trusting a joint number. On executed results it raises — points are a record-time
  fact, not carried on disk.
- **The nominal names the factor.** A weight family's third argument says which factor of the
  ambient weight it varies, compared by node: the same SF node **joins** that factor (one SF, its
  universes), the `graphed.weight()` handle makes a **relative delta** that replaces the product,
  anything else is a **new factor**. A re-computed expression is never the same node however equal
  its values, and `graphed.explain` says which outcome each call got (level 20).
- **The guard fires before a runaway is minted.** `max_universes` names the count and the families;
  turn the runaway into a one-line placement selection.